In [ ]:
import duckdb as ddb
import pandas as pd


# l. Exploración incial
¿Cuántos registos? ¿Qué periodo cubren?

In [72]:
# Conectar a ddb en memoria
con = ddb.connect()

# Leer JSON
con.execute("""
CREATE TABLE raw_logs AS
SELECT * FROM read_json_auto('../data/*.json')
""")

# Ver estructura
con.sql("""
DESCRIBE raw_logs;
""")

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ log_id           │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ timestamp        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ service_id       │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ server_id        │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ trace_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ span_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ method           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ endpoint         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ status_code      │ JSON        │ YES     │ NULL    │ NULL    │ NULL    │
│ response_time_ms │ JSON

In [73]:
con.sql("""
    SELECT 
        endpoint,
        COUNT(*) as hits,
    FROM raw_logs
    GROUP BY endpoint
""").show()

┌───────────────────┬───────┐
│     endpoint      │ hits  │
│      varchar      │ int64 │
├───────────────────┼───────┤
│ /api/auth/logout  │   403 │
│ /health           │   407 │
│ /api/search       │   409 │
│ /api/orders       │   404 │
│ /api/auth/login   │   389 │
│ /api/users        │   403 │
│ /API/AUTH/LOGIN   │     6 │
│ /api/payments\n   │     7 │
│ /API/USERS        │     3 │
│ /api/search\n     │     5 │
│       ·           │     · │
│       ·           │     · │
│       ·           │     · │
│ /api/orders\n     │     7 │
│ /api/checkout     │   440 │
│ /api/cart         │   392 │
│                   │   227 │
│ NULL              │  1517 │
│ /METRICS          │     8 │
│ /api/products\n   │     2 │
│   /api/users      │     6 │
│ /api/auth/login\n │     2 │
│ /api/cart\n       │     4 │
└───────────────────┴───────┘
  46 rows         2 columns
  (20 shown)                



# ll. Limpieza y vista de stagging

In [74]:
# Normalizar una vista saneada, remplazo nulls y celdas vacias con un valor fijo y casteo tipos de datos, corrijo strings con mayusculas
con.execute("DROP VIEW IF EXISTS logs_clean;")
con.execute("""
CREATE VIEW logs_parse AS 
SELECT
    * EXCLUDE (endpoint, status_code, response_time_ms, timestamp),
    COALESCE(
        NULLIF(
            TRIM(LOWER(REGEXP_REPLACE(endpoint, '[\r\n\t]', '', 'g'))), ''), 
        'unknown_endpoint'
    ) AS endpoint,
    TRY_CAST(status_code AS INTEGER) AS status_code,
    TRY_CAST(response_time_ms AS DOUBLE) AS response_time_ms,
    CAST(
        CASE 
            WHEN LENGTH(TRIM(CAST(timestamp AS VARCHAR))) = 0 THEN NULL
            ELSE COALESCE(
                TRY_CAST(timestamp AS TIMESTAMP),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%m/%d/%Y'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%d %b %Y'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%Y/%m/%d'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%d-%m-%Y'),
                TRY_STRPTIME(TRIM(CAST(timestamp AS VARCHAR)), '%b %d, %Y')
            )
        END AS TIMESTAMP
    ) AS timestamp
FROM raw_logs
""")
con.sql("""
    SELECT 
        endpoint,
        COUNT(*) as hits,
    FROM logs_parse
    GROUP BY endpoint
""").show()

┌──────────────────┬───────┐
│     endpoint     │ hits  │
│     varchar      │ int64 │
├──────────────────┼───────┤
│ /api/checkout    │   452 │
│ /api/cart        │   403 │
│ /api/products    │   442 │
│ /metrics         │   452 │
│ /api/auth/logout │   416 │
│ /health          │   423 │
│ /api/search      │   425 │
│ /api/orders      │   428 │
│ /api/auth/login  │   406 │
│ /api/users       │   420 │
│ unknown_endpoint │  1744 │
│ /api/payments    │   457 │
└──────────────────┴───────┘
  12 rows        2 columns



Existen 103 fechas futuras y representan el 1.8% de los logs, tomo la desición de eliminarlo

In [75]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE timestamp > CURRENT_DATE) AS fechas_futuras,
    COUNT(*) FILTER (WHERE timestamp IS NOT NULL) AS total_con_timestamp,
    ROUND(100.0 * COUNT(*) FILTER (WHERE timestamp > CURRENT_DATE) 
        / COUNT(*) FILTER (WHERE timestamp IS NOT NULL), 2) AS pct_fechas_futuras
FROM logs_parse;
""")

┌────────────────┬─────────────────────┬────────────────────┐
│ fechas_futuras │ total_con_timestamp │ pct_fechas_futuras │
│     int64      │        int64        │       double       │
├────────────────┼─────────────────────┼────────────────────┤
│            103 │                5690 │               1.81 │
└────────────────┴─────────────────────┴────────────────────┘

In [76]:
con.sql("""
CREATE OR REPLACE VIEW logs_clean AS
SELECT *
FROM logs_parse
WHERE timestamp <= CURRENT_DATE
  AND timestamp >= '2024-01-01';
""")

In [77]:
con.sql("""
SELECT
    COUNT(*) as total_requests,
    MIN(timestamp) as primera_request,
    MAX(timestamp) as ultima_request,
    COUNT(DISTINCT user_id) as usuarios_unicos,
    COUNT(DISTINCT endpoint) as endpoints_unicos
FROM logs_clean
WHERE timestamp IS NOT NULL;
""").show()


┌────────────────┬─────────────────────┬─────────────────────┬─────────────────┬──────────────────┐
│ total_requests │   primera_request   │   ultima_request    │ usuarios_unicos │ endpoints_unicos │
│     int64      │      timestamp      │      timestamp      │      int64      │      int64       │
├────────────────┼─────────────────────┼─────────────────────┼─────────────────┼──────────────────┤
│           5587 │ 2024-01-01 04:28:47 │ 2026-08-22 00:00:00 │            2451 │               12 │
└────────────────┴─────────────────────┴─────────────────────┴─────────────────┴──────────────────┘



# lll. Análisis de tráfico

Qué endpoints reciben más tráfico

In [78]:
con.sql("""
    SELECT 
        endpoint,
        COUNT(*) as hits,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM logs_clean), 2) as porcentaje
    FROM logs_clean
    GROUP BY endpoint
    ORDER BY hits DESC
    LIMIT 10;
""").show()

┌──────────────────┬───────┬────────────┐
│     endpoint     │ hits  │ porcentaje │
│     varchar      │ int64 │   double   │
├──────────────────┼───────┼────────────┤
│ unknown_endpoint │  1324 │       23.7 │
│ /api/payments    │   413 │       7.39 │
│ /api/checkout    │   407 │       7.28 │
│ /metrics         │   407 │       7.28 │
│ /api/products    │   401 │       7.18 │
│ /api/orders      │   391 │        7.0 │
│ /api/search      │   381 │       6.82 │
│ /health          │   380 │        6.8 │
│ /api/auth/login  │   379 │       6.78 │
│ /api/auth/logout │   376 │       6.73 │
└──────────────────┴───────┴────────────┘
  10 rows                     3 columns



# lV. Análisis de errores
Los errores 5xx son críticos. Afectan la experiencia del usuario y pueden indicar bugs

In [79]:
con.sql("""
SELECT
    status_code,
    COUNT(*) as status_count
FROM logs_clean
GROUP BY status_code
ORDER BY status_count DESC;
""").show()

┌─────────────┬──────────────┐
│ status_code │ status_count │
│    int32    │    int64     │
├─────────────┼──────────────┤
│        NULL │         1321 │
│         200 │         1155 │
│         404 │          325 │
│         400 │          321 │
│         401 │          319 │
│         201 │          316 │
│         502 │          310 │
│         403 │          306 │
│         204 │          305 │
│         500 │          305 │
│         301 │          303 │
│         503 │          301 │
└─────────────┴──────────────┘
  12 rows          2 columns



In [80]:
con.sql("""
SELECT
    endpoint,
    COUNT(*) as total_errors,
    COUNT(DISTINCT user_id) as usuarios_afectados,
    ROUND(AVG(TRY_CAST(response_time_ms AS DOUBLE)), 2) as avg_response_time
FROM logs_clean
WHERE status_code >= 500
GROUP BY endpoint
ORDER BY total_errors DESC
LIMIT 10
""").show()

┌──────────────────┬──────────────┬────────────────────┬───────────────────┐
│     endpoint     │ total_errors │ usuarios_afectados │ avg_response_time │
│     varchar      │    int64     │       int64        │      double       │
├──────────────────┼──────────────┼────────────────────┼───────────────────┤
│ /api/orders      │           90 │                 55 │          16328.72 │
│ /metrics         │           89 │                 60 │           92378.7 │
│ /api/products    │           83 │                 53 │          90694.03 │
│ /api/users       │           79 │                 45 │          15089.42 │
│ /api/auth/login  │           77 │                 47 │          13774.15 │
│ /api/search      │           77 │                 45 │          16958.67 │
│ /health          │           76 │                 41 │          58239.37 │
│ /api/payments    │           75 │                 41 │          14942.72 │
│ /api/checkout    │           75 │                 52 │          14981.49 │

In [91]:
con.sql("""
SELECT 
    COUNT(*) AS total_logs,
    COUNT(CASE WHEN status_code >= 500 THEN 1 END) AS total_errors_5xx,
    ROUND(
        COUNT(CASE WHEN status_code >= 500 THEN 1 END) * 100.0 / COUNT(*), 
        2
    ) AS error_percentage_5xx
FROM logs_clean
""").show()

┌────────────┬──────────────────┬──────────────────────┐
│ total_logs │ total_errors_5xx │ error_percentage_5xx │
│   int64    │      int64       │        double        │
├────────────┼──────────────────┼──────────────────────┤
│       5587 │              916 │                 16.4 │
└────────────┴──────────────────┴──────────────────────┘



# V. Análisis de performance por endpoint
¿Qué endpoints son más lentos? El promedio puede ser engañoso. El percentil 95 (p95) me dice cuánto tarda el 95% de las requests.

In [81]:
con.sql("""
SELECT 
    endpoint,
    COUNT(*) AS requests,
    ROUND(AVG(response_time_ms), 2) AS avg_time,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY response_time_ms), 2) AS p50,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY response_time_ms), 2) AS p95,
    ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY response_time_ms), 2) AS p99,
    ROUND(MAX(response_time_ms), 2) AS max_response_time
FROM logs_clean
WHERE response_time_ms IS NOT NULL AND status_code < 500
GROUP BY endpoint
HAVING COUNT(*) > 100
ORDER BY p95 DESC;
""").show()

┌──────────────────┬──────────┬──────────┬────────┬────────┬──────────┬───────────────────┐
│     endpoint     │ requests │ avg_time │  p50   │  p95   │   p99    │ max_response_time │
│     varchar      │  int64   │  double  │ double │ double │  double  │      double       │
├──────────────────┼──────────┼──────────┼────────┼────────┼──────────┼───────────────────┤
│ /api/cart        │      255 │   533.24 │  232.0 │  488.0 │    500.0 │           73920.0 │
│ /api/auth/logout │      247 │  1229.17 │  238.0 │  484.4 │    500.0 │          125392.0 │
│ /api/orders      │      244 │  1043.82 │  228.0 │  483.0 │   498.57 │          195434.0 │
│ /api/auth/login  │      240 │  1468.42 │  238.5 │ 482.05 │   495.61 │          166690.0 │
│ /api/checkout    │      267 │  1599.71 │  284.0 │  482.0 │ 32844.16 │          175417.0 │
│ /api/payments    │      278 │  1996.72 │  257.5 │ 479.45 │   2850.6 │          255807.0 │
│ /metrics         │      256 │  1235.66 │  261.0 │ 476.25 │   498.45 │         

In [82]:
con.sql("""
        WITH error_5xx AS (
            SELECT 
                endpoint,
                COUNT(*) as error_count
            FROM logs_clean
            WHERE status_code >= 500
            GROUP BY endpoint
        ),
        total_requests AS (
            SELECT 
                endpoint,
                COUNT(*) as total_count
            FROM logs_clean
            GROUP BY endpoint
        ),
        avg_time_response_ms AS (
            SELECT
                endpoint,
                ROUND(AVG(response_time_ms), 2) as avg_response_time_ms
            FROM logs_clean
            GROUP BY endpoint
        )
        SELECT 
            error_5xx.endpoint,
            ROUND(error_5xx.error_count * 100.0 / total_requests.total_count, 2) AS error_percentage,
            error_5xx.error_count,
            total_requests.total_count,
            avg_time_response_ms.avg_response_time_ms
        FROM error_5xx
        JOIN total_requests USING (endpoint)
        JOIN avg_time_response_ms USING (endpoint)
        ORDER BY error_percentage DESC
        """).show()

┌──────────────────┬──────────────────┬─────────────┬─────────────┬──────────────────────┐
│     endpoint     │ error_percentage │ error_count │ total_count │ avg_response_time_ms │
│     varchar      │      double      │    int64    │    int64    │        double        │
├──────────────────┼──────────────────┼─────────────┼─────────────┼──────────────────────┤
│ /api/orders      │            23.02 │          90 │         391 │               4886.2 │
│ /metrics         │            21.87 │          89 │         407 │             20668.25 │
│ /api/users       │            21.64 │          79 │         365 │              5373.82 │
│ /api/products    │             20.7 │          83 │         401 │             18423.23 │
│ /api/auth/login  │            20.32 │          77 │         379 │             19631.05 │
│ /api/search      │            20.21 │          77 │         381 │               4448.2 │
│ /health          │             20.0 │          76 │         380 │             12437.99 │

# Vl. Tendencia horaria
¿A qué hora hay más tráfico? El tráfico varía por hora, saber cuándo hay picos ayuda a planificar capacidad

In [83]:
con.sql("""
SELECT
    EXTRACT(HOUR FROM timestamp) as hora,
    COUNT(*) as requests,
    ROUND(AVG(response_time_ms), 2) as avg_response_time,
    SUM(CASE WHEN status_code >= 500 THEN 1 ELSE 0 END) as errors
FROM logs_clean
GROUP BY EXTRACT(HOUR FROM timestamp)
ORDER BY hora
""").show()

┌───────┬──────────┬───────────────────┬────────┐
│ hora  │ requests │ avg_response_time │ errors │
│ int64 │  int64   │      double       │ int128 │
├───────┼──────────┼───────────────────┼────────┤
│     0 │      490 │           4169.97 │     78 │
│     1 │      213 │           4572.25 │     42 │
│     2 │      201 │           7347.51 │     37 │
│     3 │      213 │          14587.41 │     31 │
│     4 │      228 │           4142.28 │     38 │
│     5 │      230 │           3127.88 │     31 │
│     6 │      247 │           3407.81 │     44 │
│     7 │      232 │           6316.06 │     42 │
│     8 │      209 │            3491.4 │     34 │
│     9 │      226 │           4411.01 │     44 │
│     · │       ·  │              ·    │      · │
│     · │       ·  │              ·    │      · │
│     · │       ·  │              ·    │      · │
│    14 │      236 │           2947.93 │     35 │
│    15 │      201 │           3666.95 │     31 │
│    16 │      214 │           3196.44 │     33 │


Top 3 requests más lentos por endpoint utilizando window function

In [84]:
con.sql("""
WITH ranked AS (
    SELECT
        endpoint,
        timestamp,
        response_time_ms,
        user_id,
        ROW_NUMBER() OVER (
            PARTITION BY endpoint
            ORDER BY response_time_ms DESC
        ) as rank
    FROM logs_clean
    WHERE status_code < 500
)
SELECT * FROM ranked
WHERE rank <= 3
ORDER BY endpoint, rank;
""").show()

┌──────────────────┬─────────────────────┬──────────────────┬─────────┬───────┐
│     endpoint     │      timestamp      │ response_time_ms │ user_id │ rank  │
│     varchar      │      timestamp      │      double      │  json   │ int64 │
├──────────────────┼─────────────────────┼──────────────────┼─────────┼───────┤
│ /api/auth/login  │ 2026-02-13 10:20:32 │         166690.0 │ 4064    │     1 │
│ /api/auth/login  │ 2025-08-01 00:37:49 │         127155.0 │ 301     │     2 │
│ /api/auth/login  │ 2025-11-29 22:07:16 │            496.0 │ 8703    │     3 │
│ /api/auth/logout │ 2026-08-08 00:03:14 │         125392.0 │ 103316  │     1 │
│ /api/auth/logout │ 2026-04-19 10:02:22 │         120696.0 │ 387     │     2 │
│ /api/auth/logout │ 2025-09-02 00:37:34 │            500.0 │ 8146    │     3 │
│ /api/cart        │ 2026-07-04 04:40:31 │          73920.0 │ 2559    │     1 │
│ /api/cart        │ 2026-03-28 02:39:15 │            500.0 │ 6341    │     2 │
│ /api/cart        │ 2024-07-09 03:06:43

Comparación con periodo anterior ¿Cómo cambia el tráfico día a día?

In [85]:
df_res = con.sql("""
WITH daily_stats AS (
    SELECT 
        DATE(timestamp) as fecha,
        COUNT(*) as requests,
        ROUND(AVG(response_time_ms), 2) as avg_time
    FROM logs_clean
    GROUP BY DATE(timestamp)
)
SELECT
    fecha,
    requests,
    LAG(requests) OVER (ORDER BY fecha) as requests_dia_anterior,
    requests - LAG(requests) OVER (ORDER BY fecha) as diferencia,
    ROUND(
        (requests - LAG(requests) OVER (ORDER BY fecha)) * 100.0 /
        LAG(requests) OVER (ORDER BY fecha), 2
    ) as cambio_porcentual
FROM daily_stats
WHERE fecha IS NOT NULL
ORDER BY fecha;
"""
).df()

df_res

,fecha,requests,requests_dia_anterior,diferencia,cambio_porcentual
0,2024-01-01,8,<NA>,<NA>,NaN
1,2024-01-02,3,8,-5,-62.50
2,2024-01-03,5,3,2,66.67
3,2024-01-04,7,5,2,40.00
4,2024-01-05,8,7,1,14.29
...,...,...,...,...,...
953,2026-08-14,4,7,-3,-42.86
954,2026-08-15,1,4,-3,-75.00
955,2026-08-16,7,1,6,600.00
956,2026-08-17,12,7,5,71.43


In [86]:
con.sql("""
        WITH daily_stats AS (
            SELECT 
                DATE(timestamp) as date,
                COUNT(*) as requests,
                ROUND(AVG(response_time_ms), 2) as avg_response_time
            FROM logs_clean
            GROUP BY DATE(timestamp)
        ), 
        daily_percent_change AS (
        SELECT 
            date,
            requests,
            requests - LAG(requests) OVER (ORDER BY date) as difference,
            ROUND(
                (requests - LAG(requests) OVER (ORDER BY date)) * 100.0 / 
                LAG(requests) OVER (ORDER BY date), 
                2
            ) as percent_change
        FROM daily_stats
        )
        
        SELECT AVG(percent_change) AS avg_daily_percent_change FROM daily_percent_change;
        """).show()

┌──────────────────────────┐
│ avg_daily_percent_change │
│          double          │
├──────────────────────────┤
│       31.787011494252877 │
└──────────────────────────┘



In [87]:
con.sql("""
WITH requests_per_hour AS (
    SELECT 
        EXTRACT(HOUR FROM timestamp) AS hour,
        COUNT(*) AS request_count
    FROM logs_clean
    WHERE timestamp IS NOT NULL
    GROUP BY EXTRACT(HOUR FROM timestamp)
)
SELECT 
    EXTRACT(HOUR FROM timestamp) AS peak_hour,
    COUNT(*) AS request_count
FROM logs_clean
WHERE timestamp IS NOT NULL
GROUP BY EXTRACT(HOUR FROM timestamp)
HAVING COUNT(*) > (SELECT AVG(request_count) FROM requests_per_hour)
ORDER BY request_count DESC;
""").show()

┌───────────┬───────────────┐
│ peak_hour │ request_count │
│   int64   │     int64     │
├───────────┼───────────────┤
│         0 │           490 │
│        18 │           251 │
│         6 │           247 │
│        14 │           236 │
│        17 │           235 │
│        10 │           234 │
└───────────┴───────────────┘



La hora 0 esta inflada debido a registros sin hora que el casteo puso por default en hora 00:00:00

In [88]:
con.sql("""
WITH requests_per_hour AS (
    SELECT 
        EXTRACT(HOUR FROM timestamp) AS hour,
        COUNT(*) AS request_count
    FROM logs_clean
    WHERE timestamp IS NOT NULL 
      AND EXTRACT(HOUR FROM timestamp) != 0  -- Excluimos la hora 0 acumulada por defecto
    GROUP BY EXTRACT(HOUR FROM timestamp)
)
SELECT 
    EXTRACT(HOUR FROM timestamp) AS peak_hour,
    COUNT(*) AS request_count
FROM logs_clean
WHERE timestamp IS NOT NULL 
  AND EXTRACT(HOUR FROM timestamp) != 0
GROUP BY EXTRACT(HOUR FROM timestamp)
HAVING COUNT(*) > (SELECT AVG(request_count) FROM requests_per_hour)
ORDER BY request_count DESC;
""").show()

┌───────────┬───────────────┐
│ peak_hour │ request_count │
│   int64   │     int64     │
├───────────┼───────────────┤
│        18 │           251 │
│         6 │           247 │
│        14 │           236 │
│        17 │           235 │
│        10 │           234 │
│         7 │           232 │
│         5 │           230 │
│         4 │           228 │
│        23 │           228 │
│        11 │           226 │
│         9 │           226 │
│        20 │           223 │
└───────────┴───────────────┘
  12 rows         2 columns



# Vll. Análisis de errores según método HTTP


In [89]:
con.sql("""
WITH errors_by_method AS (
    SELECT 
        UPPER(TRIM(method)) AS method,
        endpoint,
        COUNT(*) AS error_5xx_count,
        ROUND(AVG(response_time_ms), 2) AS avg_response_time_ms
    FROM logs_clean
    WHERE status_code >= 500 
      AND method IS NOT NULL 
      AND LENGTH(TRIM(CAST(method AS VARCHAR))) > 0
    GROUP BY UPPER(TRIM(method)), endpoint
), 
ranked_errors AS (
    SELECT 
        method, 
        endpoint, 
        error_5xx_count,
        avg_response_time_ms,
        DENSE_RANK() OVER (PARTITION BY method ORDER BY error_5xx_count DESC) AS error_rank
    FROM errors_by_method
)
SELECT * 
FROM ranked_errors 
WHERE error_rank <= 3 
  AND error_5xx_count > 3
ORDER BY method, error_rank;
""").show()

┌─────────┬──────────────────┬─────────────────┬──────────────────────┬────────────┐
│ method  │     endpoint     │ error_5xx_count │ avg_response_time_ms │ error_rank │
│ varchar │     varchar      │      int64      │        double        │   int64    │
├─────────┼──────────────────┼─────────────────┼──────────────────────┼────────────┤
│ DELETE  │ /api/search      │              11 │              16671.1 │          1 │
│ DELETE  │ /api/products    │              11 │             14219.45 │          1 │
│ DELETE  │ /api/payments    │              11 │             12514.64 │          1 │
│ DELETE  │ /api/checkout    │              10 │              14360.8 │          2 │
│ DELETE  │ /api/auth/logout │              10 │              18100.4 │          2 │
│ DELETE  │ /api/users       │               8 │             16185.75 │          3 │
│ DELETE  │ unknown_endpoint │               8 │             14291.14 │          3 │
│ GET     │ /api/orders      │              36 │              172

# Vlll. Exportación para generación de dashboard

In [98]:
# Exportar: Tabla completa de logs con campos derivados para Looker
df_logs = con.sql("""
    SELECT 
        log_id,
        CAST(DATE(timestamp) AS VARCHAR) as date,
        EXTRACT(HOUR FROM timestamp) AS hour,
        EXTRACT(DAYOFWEEK FROM timestamp) AS day_of_week,
        method,
        endpoint,
        status_code,
        CASE 
            WHEN status_code >= 500 THEN 1 
            ELSE 0 
        END AS is_error_5xx,
        CASE 
            WHEN status_code >= 200 AND status_code < 300 THEN 'Success'
            WHEN status_code >= 400 AND status_code < 500 THEN 'Client Error'
            WHEN status_code >= 500 THEN 'Server Error'
            ELSE 'Other'
        END AS status_category,
        response_time_ms,
        user_id,
        user_agent,
        client_ip
    FROM logs_clean
    ORDER BY date, hour;
""").df()

df_logs.to_excel("../results/server_logs_complete.xlsx", index=False)